# NOVA SN68 — scoring Boltz-2 sur GPU Kaggle

Note un lot de molécules avec Boltz-2 **exactement comme les validateurs NOVA** (même entrée, même MSA, mêmes paramètres, même graine).

Notebook **autonome** : le code, le lot (`inputs/kaggle_batch.csv`) et la séquence de O15427 sont intégrés (cellule de données). La MSA des validateurs est téléchargée via jsDelivr (GitHub en secours) et vérifiée par empreinte SHA-256.

**Réglages Kaggle :** *Accelerator* **GPU T4 x2**, *Internet* **On**.
- Test rapide : mettre `TOP = 2` ci-dessous puis **Run All** dans l'éditeur.
- Lot complet : `TOP = None`, puis **Save Version → Save & Run All (Commit)** ; récupérer `scores_<RUN>.csv` dans l'onglet *Output*.

Seuls `scores_<RUN>.csv` et `boltz_<RUN>.log` sont écrits dans `/kaggle/working`. Usage de recherche uniquement : rien n'est soumis au réseau Bittensor.

In [ ]:
# Paramètres
TOP = None          # None = tout le lot ; ex. 2 pour un test de durée rapide
RUN = "kaggle_batch1"
MULTI_GPU = False   # True pour essayer les 2 T4 (plus rapide, moins éprouvé)

import base64, io, os, pathlib, subprocess, sys, tarfile, time
T0 = time.time()
WORK = pathlib.Path("/kaggle/temp/Nova")
VENV = pathlib.Path("/kaggle/temp/bvenv")
OUT = pathlib.Path("/kaggle/working")

def step(msg):
    print(f"[{time.strftime('%H:%M:%S')}] +{(time.time() - T0) / 60:5.1f} min  {msg}", flush=True)

def sh(cmd, **kw):
    """Run a shell command, show its output, stop the notebook if it fails."""
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    out = (res.stdout + res.stderr).strip()
    if out:
        print(out[-3000:])
    if res.returncode != 0:
        raise RuntimeError(f"échec ({res.returncode}) : {cmd}")
    return res.stdout

step("démarrage")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# Données intégrées : code nova_lab, scripts/boltz_score.py, inputs/kaggle_batch.csv, séquence O15427 (tar.xz en base64)
MSA_NAME = "O15427.a3m"
MSA_SHA256 = "dd247998bf28400fcd4551557b76b46ce6f9c532f5fc233ad01974f3574090c7"
MSA_URLS = ["https://cdn.jsdelivr.net/gh/metanova-labs/nova@db9bf190a6907338a05871b9658f42a089229675/data/msa_files/O15427.a3m", "https://raw.githubusercontent.com/metanova-labs/nova/db9bf190a6907338a05871b9658f42a089229675/data/msa_files/O15427.a3m"]
PAYLOAD = "/Td6WFoAAATm1rRGAgAhARYAAAB0L+Wj4Mf/KbZdABcL3ASAUDIm7h19G/PyyzPs7Pg38OrWeznBvBtWH3VunhnX8ZVi1dutW8VtrRSGQpkg6gBUZhYzjFPmRgpt8SqPHbBm5QrtYza3OF2Ia/CnoeWmEQjcYwsxyfqr0U3wmVmMFaP7ZA0s2uQAEYTMBk8lB12I6xuqcRE0oLgD4yHJ9uKW8CrUcT4rsIvxp+Y69//pvhaLJwgbHRIZd/AVNxJAYQIgqe0AqMBGwOJxoEck7uIbeYRl8QRGLu8IOijG1NfHjN8BR3Wdd076gSk9T/X7bD3qLwa7DmhEnawiO2/yWNZK9kQ70k/Uv11mYeI4v7zmeOjBmFQ8zyusM0Kxc2oDYe46M7ld5kLsWXLdnjaVYiHVfRn3hyZUZu1/BJDuzyEZ2qKK/0iInpMtYF7J4EmAnMT9JiyFMxgToKvOWvquINspxi/UyEkvEdhcmxE3nTz0LHSAJbgmzXBhNiIL0DTJh+TotSU3qLmEKu3EXbITyvbJr6/rS18R2wSjmJLfqa7FNN050joM+XDYG7kEmVC8JtGe4TVhmmBdDWi+ghkXirtZH8pvIbJ6jT//Bc/kvKDLc8iyYfBss9FTRNkoGVo3i9nHhNDRXDneft78XUZQXOKBdMTeXksiBlf/H4ptI508oQLbnhfbZVI5FP42d/nsEO5WtLONKTM46pShUDfg3kaDebinba6Oeb/Vd09wCQiLmxaI6R5FQnGzOQkKvyjQtAH5UXLMOc5LhBor5Ja+5yT1ezWDM22dh9ls6yG0xtkNKMrerBuThZ39s9u8CkVzLzWuQbNMEL5ftclgsjxvMkxB+lf0dYUOp/4xHnzqMjrmmd9ChrD20ZVwD0Wi7i2dClR1oeDkW98rNZQXpB4R/0gJqOFrsVU9n1UuXZavtUI+AIbQpk89pQ/ft0AMPU3sg7xuwhH6aULWez4p4nnufp0jNyS06oYQhABvMjjtk2SNloi3n95+nR5xUYrPMHVD/g3Uikr8aRRjK3alR6VOoRdUi4TjbnLH97yfPOlBAoqIzm+TAuaA+QtZMOwjrhKvHCVgUaX/Vr9JJ4xYCldDtYCqGNhPodayWMH9w74/G0iSYzsska57qTa3dB2Lu70+2RR0OgstHPtrxXM1q7doSHX+YLcrpuOeqMX5hDvOe403KuPA0AAy/iqMzdKowcLhcZSPFayO3KyJdNxeAbmUt+/0OwfXEte5PKWA7ERl4DKIVjMWOW3ekFJOyIjwzPIarPxg1xJBTp2VY94T8dZJYwCcA0zDDb0QqB9y7kN1U8UD21sFN769BV3zCqPkXrYh8N2Rau8OoyaJBjFbzC6WqbBnoYVVsQ/vh/FsMICdNUsBiXbmTmOdx3bWF9Oxf7boa9AXdsBl0WylZiVwO+45f5/D67Jjzex1DFSLTC8ErsWUXCFrAtxOZlTUT3sGUjtZxt9kF8/i++xid0s16R3RA3od3n/POml0hy7JEK+uNP9PxcCg29NTHdsC+M2XkyaQfU0SzIkNs8F7KhTmU03ePOu7Xzst+MoHnHERo8NG/7j8HilfDeXW1pQpDCn+lwh7Bh3TYPG3BHLNeWMulCbBM+i0Tgu+bkQl9ktszoULnIpW04ZGhtY1yAfqFQktTnA7zJQ7XHFYbtCvgWAFFhJmRSEtOKIcTMZgNZs2kGN+mZjygoY1lgGFTaoEtDKD2jGFGJYoqB5JR6BNg2zqyunSYfxFMrznAaVibSNGbhOsyKY864LGHMQPjE4gCzyD9VGd+Thtd9za03M1TwoGw0dPbRNixjKDm+7D8eBNQM/ELm+sDuB/osWz/BcwrgrHFOmIBG8JpxlO4HjJuiPE82e6j7zQwc29T7oSdTsi/K9RxSrOcQkwJ/Kl8PZPkLgqcRW4QBwC81vTJPpseVnOuN30oAMynD17wyK/4+YS5KjgbcEFc0wGI5vyj/Fzy3lLp3tqQ9L2jH2a03Hk0FuuNZ5Cyf+t6pp40lYdwft+6IGf33zxATOeCcPijilY+he07A53m3BboYF0rDLEQvMJGL+P7+troQJwfe7n4NE0Lj2JX2Lh7C1qw1g62Qlpp8g4faxNud5dwGS/Dw3XzquMNf7q+j0Q20IAgfKZzqOIbrbyw4o3epc7MkUWrKbxw210xLu0uC9/N4hkYRyOrASOKBSH6spwDsiJzH/9pjsq2QWy9SVRF46CBZhMYfXV7Jx4pEGvYBOEaN2KNSYLTzptzHXIqfWOxNkQHSeatXr9PoZApkZF7dg2kFW3Y9Z32+95O9LnqyTZRhikiFW1FrPb3O5xIbzUTCI+IoB51NkHfdMr6TRSKPCo9hivDk7It0NLwfF9yet0uIqUUOzOgnwo+XXSRb7HQqdc8lIoKKZp+8j4t8x1q1LXjSCpi0almHHfoFHvd0xYyuEHz6L3BQNE4WwAjQ98QC7es9jDI7xIj0hGX5ZzWkZE68fokBPVB6fwm+poENHe/5+tQl5P/x6f+X06b+23HgwP2Y/yWvags4kmxVdTU16E8g1CJIBCJlXfCQukIRKImnJJqk/hrs7Wi7Nm/qzzENy5vEm18rKVA2GSpGjVWXTjedcJE/ZuqRtz98I6SitIMBWHYML0GXNkgJ+Ybu9LIwLsiQ4nD1RmLR19xw8kqA0Y0KQHISJ0CoKrbxJGsyR7AxU+w+cViq3b9QxXp++I3PdbOGp4276d+0Me9iuq9xEwJzUqvwylCNLo9mphNH7syT0lHKCqOTPbddiSFT6UM5GrPXQAkBEkmFj/wsAqisPQ2xVNo0WeJxJ2yVrPlLv2j1dLLMpCWhe8zXt7+82lKT/mg9csyLoSaqwJ2g5pa5Gn1TqbF6XSZ4qIW7BNGkZzfqq9Q7HduJWsNk3FP6ENlTOuPHb+cTPz5XUnTkgWmtfQF3XOClP3zUS8fh9bnaIGjRwp0mWwbo/HCpHDI5FtAGNs8Uvx0tmRrofXVpon8hYe+JPxSDWHPalbRRYXBbAIz05vQj+75+uhnIRCT6yHQhQMKpSMJHp2PRRTHEx3svJOZv3XEEQACfDP8O4tas0q+DaR6vgkTFdoxGZQsLsPoWqzlhWYGgSk/GiI/52RWgNjB4dhXnnxURBU2ihpO1AgVgX8AQpBA3E1TJzzNPSwJhSaL7XFpV1kppsfbhCpFeluCVA1Rh9y4UWldeY07iubU86wJDtYqx/X/YYbsbgznCgvNzPMQyP8T/Fpn2IS8DQFg9P9EaLQKqWgzCD4JZzNPD9FX5rasP2NiC4B8FeukWQjqcOkgERUITcCZROAJ2R8JPSyrPi+73HiIp5PRAzvfzE1PZ8V8kqfDOgiyQHLE1ItadiF0qUJNUtbgMopTvb5s8FbKk8FIT6W7VubrK3+dfvSvSMt4cnNePbTrP3HXyuF/Kin2RBMmQebzfr6CslkLSJHEN3scwhobx45w0bLIm6HGZQZAgGhdrvFHeTH4AbgBD+xjBR6iOjI1AQscqipJmDgzm9qlca4oIkpmyytW0wQlIss6St5HUeDHSnzW1oj61kHB6GP8v67jy5rF3wLNxfCfJMYu/bfFh+WSwh3qlaZGqOgchkUL7sKKpVSguobhhexDVkPXKc1uvXANT6hyFprYx5GVqpSjExD20L2svvOQHI1vGk6Fw5cbsLtquTo66tqRFwudiBCYhgY0buykGaX6cu+v8GpHAfnKFY6C5ulPN3gDTl5iNcBxBuGKQolvZqF/gcWDcI5Ktx5tEErq0CM3jxdmwpR+a5ctUNmym+36OOqBqvBOWWTZbiwtWuXPUV7ifS7Ixtoo8GcXaYXBr4fL0K5jeS7/x5TY19MtuG2IKqo9Ndl6RiHanYHW4YJe+bQj5JpkvKKsW+SvagkOaLeTov3aHCFpOp2OqLXSwYQQHGv1CL1s6Njl33h6gqfaXbnC8I2Ho93bne+6oz2lONaNs4pUjg1nP9kB9rZ+Y8gvP7E3PRyO2JpzSVY0R7i1ahXwhMjW7WqJV6oBaMUU/hjz8UaOflibYbMcUGIpUyzLMFuaGLh40TQTv9fH7NEz5A5STY7bl4g6l3g7wWjn5IeUmfVEO8yDoWlFSUVyJIDl3WvFzeszpVwSE5ZBKNEzUS+L7UyazfPiBQDmDt9+E0PUMWkuTgg+7shP5WCVEdrqXktyOLar4Hc9VFnhX0lEFmXpd8AnI1ZPcD81PuNMd0D9cOfsr8aG6joa6ZXMN7I9pS/QiObuLUpXHHjI/BYEPR1m8Pe24jBJv2wHqhXluuc28PSbQNbzVOs9T7urqfxWX8BwUG9IitagkhOVnPlKMBe8JYdt5PmonGVd6Dk+WK/BMBKUrM4oCk0F3NP4U3Ri+EhDjbMSZLXokLKDXO4z+7VP0jQCGctmnb0Xj6U6Ek2UGc+jVai2XpYlF4zOWsVAzo1h4iYSNirkSDXX2FtF5J+czLOS4xZX9IegtjGx0oEdCKeMdCcfEpBv5pw1b3G6nFrnI1Slp0R3NlaUAEWHCezt9PTNNsPp1Kxs1b2KaiwsNrRvP9y9Z6tb/sBtM5OVugylryWZkzl6G6MMelRc07a0XoJq9EGOIXq3Tf4pWZxof46KCiAITVcNO0OAy+66rYTOs/2A9IDGiOHxrSfIIGpeS15uQXPJEXU20TjA/R11iC2Pe5ZCoyutGXncTqdoxpgvmRaY9NsQgRbZ3SU5I9EsI6+ahvkqqAU2FScVwvQamJGluY3LQidQOeRLk87ZTJJwsYwx/2UYQQbo0D7qhSaZ77kS3L011qdPQvjsKJjA0vB52SH4qx76okf/xWQPldmcj3UWzs8faJ2IJOXy7BFCyMF55RYR9qtSE9etPPlYu0M3u4P+mK2rn2sPnrlji9nTPy7f/tbYRcmYK9xQX58cockcqd8oqnjXx75Vx2VQYSO8WBXU+kMg+chuomoRcu1+Dhfiko+XB2mF70WgxbxTnnDlejbHa1IrXQDMZXVQ4hY03kW65nGnoiUynN3bGBsd1xGl9MHWaSj3QxgnQfmi5tG9VzZXekqdw6F1em7Vciz0Sn8lYFDYKj74tlFdhOcDczyGGkLV5y8dpwCiksiBYe3+vsTi/AVj2xeHrP2m3HZT4stpn79Kxaj6boebTLM07rWvxkSxRKLhtQUwxkT9zen1xV9Qm5xz2IToJOk/8n4BAYLsShBmjgY1JuItjQq7zTh+IqKA4V3PPPH/UPhgGnQRRx1u8nkbK5VBJ6BzaEoFudFCASBiCS46yA8MZnTf3nXA1SPQAAbzy9YBNgjnkjLpDPkY6GUCA8Sxa92Q3iKTXR948oxCupZv9julCXhYYBGo2JlPtRRwXmpBFkqyARNdgFZn+KajWxp1Q64klF7a80SFHzBuJLeVx8aNlM5olGjBsjCLNk4MArti5tbFkxxjOsMlO6p4HzM5J5j+/TJaE37Eo02Wx3tjZUZwynt1udLIUX2XleQwOMLBgKlE0mZug4RlVlVx/tVHcCSCVJnS0WWCZJDK6SjQUGC1kvUWD43p49XOwZQejyxdVe2ims/NMgulMKYw4ye5fzCC33fpzFbyC2dMo8atL9YdlZr/UfHOKMOKPoSr8SQ7p5xfuicR6aa58cyfLMz+tRjEnSFQ+3cVblAlvz0va359H1qTOqtYaWxmlAa9CwvD6bdratGyJXwR4xqdFaou5wMgrTNnbVGqk0SaP25Fxun2FsMRo/vRtDB556//duf5aunCeR5z5kINRj6I/LtiNOiPyfJoC1nD2Jdsqizf8irGV+R4699RTZlpMpTzwLBg7NS5e5/tS8ebH6oLAR+1ie7a81Gyes1pNQQ66feWEeEISld92CVROPN6GDebVG/ouWNOTjyFUGOCOjCeRe1LeHeJswwjL8E4toCC9KrKU5cXTbvw8+G/Yv/0pH4j4e6kK1IyBLRwlfLFpF8nBX0g42jN/PzjC1KOBs6P+gogHaHnHMGX4lgxJjFqml/fpdwGJrjRuhm6NkKpn3A6uRGw90WbYX1UqtPbdNKsYacVaedVh7Wn6UTb/1xGCfjNZ5Rhv2QE63E9wAwBGTuvJLArr10ELDgeYEVEc8nvZsCYR2Pdgphi/tVDk/JocjnJsyVF8+tNujk3FmSWQqMlscqe2ERyQ+bDxXDQKPrVmxbQ4yFL4bEVX5o9cI7qClQODUCXXE8hxgWg0b/hSVMX6nhjjUndXSRJyhqqxImNwCwy+lPH1QaZ6jz4EKVlG9JXFy9n+AIlZomwnIggDio+97IVrPkcemYYXqBJ6OdyiqTU8A0t/TnlWbqjbLPbqwneVTWBiDIaqrexIB7P1lRCLBl3qmqHkTlNjVj3JlW70fk6eus+rz/RULRDOgcDjrt3/fXVyAeg/bW3qd526jkjXEZxl5Kj/flx0uD/k65d3pscUkH4T91rDhhgN5Rf8JVNfP242D8y/R6E3lf7wpc7DsCmRM3TXSIBYMCtQ1Lfgco591UMg9k0PVsqMpvk5rHCw84ywVr/MZI+6liExpaVNFmJGM1RmAwEk0g8Tf3LiwmTMpG+2rqiMd8rFFW0JQin5mIVY3txY7epDRxN8Jxg+VCyuTeDFgtyp4Cx/eUwtfiuKdNLXJgwZbQXJY4e1Zn5doLbcWsbFtwHIMrTOWnk5/BsOd0/u614Nh9aVS32iPs9zW6XaH7HwqNxlN5Enay7UAtHKa/ew8pTcB3CdlZw+225Jo/+/9wubA4sEYdF1foEpm2yxFfBdsf8q778u/O34kM8ZnQnq18RLOcKz6tX2ibb5HI2AOg1xD2kK/90CdfepiJQA31gQjs1C4GLFJW1nJRcKPfcIds+3xIxdUCMxmSnLdOGyO10guFJvnissHliGpEqc9ArWKMPHkL2q2jUkDtSDmkaTbHhm7/Uc0BFQbQvWtICMopOjFEzg2Xcw8FkhD6TnOVLYate/KRCkZREH5aPuatgK8W8+bVeh/5R4bK0uPtlHePTFMp7QW7togufgJrT9azKzMmYUKZecV55ER7g7s2FqGc2EBqJiLeMhj0kR+2ujNFqGfa7zLRqsgiv2n9h6rFR7us0iQVpLlp8IxZ6AneewI/6sZctLHGODbh0/reF9R+JPMk5EkWthSC+4/UXr1U5XrNY4uZK17/cJHBPHX11u2iV+WvMcA+MvM3/Km2FUQctw8Elw2UHrLWbTQ6mcPXyEZz9gVzjSaynBuDxD2vu4jekbdzaUptqxCCY9dcKKBS1Nr/GxFyHfj1pAxksWFXpo7Hk39C287lR+3oxeOXaokX5pQhwO/0zXz+NFztm2KtEoMKxqmjy2R64xhTuME664vgLMCUQtRS27Ph4aJxGHbj/oe/DmbgjyKR1UUEoWZr1KYTWJIeWkw4PmCMQjCzfZArWZj6Gi23C788tzkoLhWFc8BvMS+OYHeEHfbdtxJPPo9EBH1F/4/GfqYStY0Fvhg2983EszBuENUZTjTC7dNs0JlSZAOV7trOv7UlfuB3IYoxMqjf2OY6FPm2lpdvRMbUS4dtuI+yPx0J92bMl5dfJoQgIAU8RPIrerWbyTr+Psc6v5ZWF0vHoX8z93F/ddhCL6bxFgcSHioCotR3JYyldGAbs1fD/X+kSUwrTha5b2tdCeGQzqx6P3B1NJMr/ZTG/kPpTKPmPf5ceBV2fTIiaTGr9MJU3AdxFOcmMXfLumhe6p95/4Pxye5uS2p3RVCc2h37GJ/gDPsPlTtNEjOmeBYeXrozuWOaY5/9X9UsmTq3KdcIo7Qbu56PsDy7Z2H6XY0PAVGCgH1j2iQGP0pg5JOJOGdbw4q/YlNQDzVFCC/0LwQil9YKjC/poywBoy5E7SOf5HeCMhBJctFsIMfYpOBoMQ29UfdA5XLndIixAG6K2vsqzwB88EWqUeYU9U4MDM5LciIv3z1u246X7KC06LEZf/e9G8AoM3n/f1MteVp/LJbxZiKuSXmsslYg6+/Dv31RfxeOh9DE2QAKimOgi76t5TUgN/K47TRplbnkq5qX4DiUs6BE/PUiV9lARfIQi328Sjzow9+0MpeVPCNa3Gg1pJAo5OImCjIIC6QlFndX5O640wK046PJ80m2cTl667zHBC4I//SJM8Rc0mVPGrdkbOrh6Omq89HLk/zHufnwzNG7etYNkrij9sgS3IpjHHigbaczRMWVqOoPyznRIKgEcdiLK6bgmqittsM/qOW3Uc2i7DeZTW37Alnjp4jdi+RVGtLBk1j5HQMktqAY7uAninmnIEwb4OabZ9rGqoGzHvhw41gm7Im/kY/oCL2oHFnkD4MA/j9J+8Zraqo5HWlOfAaCZLkTAovT4ybafeimPefLuEfqPhqoHio6GBJVOkuAR5LNamLHFcQVgXJJO5k7/nNL6m/+8d/aes3PbyqQdWLpqB6BRMb2ZseYyl5xA80VNjJ/6XtFSpp2KsZ7ibwJUcQ8aI4a2u1ErMeROybbGhztZLTU/4MST4jrhwfDOoYZJGJDf/B+utyKbjcOxfZWIrC1nwhbDIXv9Oe+HKFe5GkbRMq+vfP4fNdOohI9pjfb8XTjgfuXYVVwu4oWdHb3ZRWEXFgdV7lxyqjqovdaQNHUl8P9MCKNkzbjBQy50ph0HGtC0fAVH+sw/lbJHDXQOLRWFn5NYHV6wIKBgET60kvfAYfwIL67XUHAtuheiFw22TT8IDpnytAtN1hCxifAD3081sjZF8toy0h6VuRuTQj1hrXMrpuPdz6Ofa7vHphyCgNN6AbcbMAJnGmIjDoPBxytKbUK4ZdWuu9icuLkQhvjsEPSA7Fb1V8pFmrWXZlOYkKqdLV9jtM3EEfnkLfq8E8SK3ZW3BYB/N79CgkmdWip1gpZNRdHroNxycUHazB/jJI3djnSp+NNhxWvuZnjHTJGx5gnu9XY/Saq21VHOOOkRxptBh9zDMMHxkO4fNhjjM+o55EpGOEZpnPLmoVTWbg5u5uZWoJLcEzSRM5LqWddaYlk8r3b2lG5znHje8FZckwmkq6DTglwvU/dCaPgO+OwlCYJ4Yzc9ZX7uwL050tkDLZtTrodU1j0GD7zWm36iIO9xTsR4bSmj9t1HFXf91EXcgzAEfiOI2XtvuTr2631j0WE7Lpe52mar1NzMcLLre0Th0CPd/Y/8yiiLXKIX3MtEHttL59w3NDfXeHj+ByvPX0ka512gUInn4bjCWMmDK56IrrTJGj3MKlNEqRzMDJ7OdcSllqti/Y0IP21WXyR44HJ3o71oyTLbPtGjmbUvuqcjX6ufa46XteI5S++CQxDFwCAzAJ6w94Yt/JTz1bli++cd78308qCGj/WBZmz+fEG+8vNqqFV+53eBctu84MGllgMMyLIH94mPDnvzVswXVYnlN4ujP4Bcka4JgjF1keHZvozYtRhUzZseWPkoc9p1fvCKPF+BgZgLmFlofiwyS/at7J+ut3TibzN7Zvu/71rV6sv7YHG5ctv01L4A78/M92Rbc+RsKKHl+ey5ZVFv6PmnSjdphFTRCh3b1Z2WhuXfYDmgH9G/P+KtmodI9UijF6jDtTuuDIc0sYGui4oNzPVbI3pDY+P+Nr3QBvKhIAtohBXudvbABFeDcSTBNEujwqgbB3d5b7qpiH7wl9nwnirAQulJQpwIqE+K2YfIYj0+gRVIXgPMFvHT0QyEgRr7hUMzrlHMTJ6ZfQBXTvz7/yvAjSl961dMq/htrTS8GZ9+ck3R5pbDPDXmg4R9WJtuRSQRZh1SwlsicTjWYW08jvtrm3EHXgF7k14vv5wpETIHatO5WKvMcW+qlpVtq5GclaO65HM2b2/KBAEbCERYelID1QBHO87fv2tn/C2P/OOU+4qJayaGLIsiNekqiTtwRGHOIjIxEJT1zPNfXtQernJE9cIXzruU85ZJnJlaEbjOVHuaF4XsqXIrI01gJho73hYp6JH6varDbRFbI3grNzNP8jE0ddR5o6fll5jpTYYM5kcsEcNJIRgoVOSXFHpwDMtfveY8MBXiwdnryL65sW83o+7Wo37EXeV6ydeSQknTCQBx+NpK0wU5CgqcaETj3Uzvi6vAGtbQpg8QRD3OcWvyaCQcsl3OmLQXRkdvYPy5PAOOxBZBwd+QHfTNYzF5KfIwsuFdPCxe70ilkcxW5Iiw0y4THVt+/9zvQc8KZzlvtZSLfh3o0inQFzUZxrnUVsNxd/Dov7wZQzC0Ncxs9AVvxkCj8xy7YuKSuHMdfV9J/tswShJ0Q83wnH1lcmBz5LCqJlQwlh2DkRlfuMvQfmQ1Qsd5qqUq4WK+mj8H6HvdVGpOZEpp67XTxKtvUv3UP+B0PHPyrd3nhUdRMTpX9U0IZJA6oV+5qKFXdaFNl/wtqOeW9Wss1FOKT0coX/KbwVzwxAVeHBcGCrK68afuioHAcXB+QifmGPUHGXx3CSVFWmWxVOIRIs2JaqDfFTjiPYo9CCSKtKsHci8vCO5oDky3YXjJDaNIRd8LZRav/LCLqepo+gi4xgSBHwb/mDa2hk3LxPlx1PIFiDiF/l4smmaSM4V5dnI6CdOeNo/ESCh2OIzsbw3nbjY8Omtj1iWMHLnizIQFy53G+WfmNRKx/Se2DIOJDdOIopSrIEsQkFuab0kmVhA2mFS7uuR2ig1JxO6oCte0xyNYk5Zz2IlIbVEbKXb5FngFEhzEejfdRfviff/finbq1nz2rmjcmTdCakFkrh9Os4Mai8P2t1Rem7ilulbSNKtyuRuF2aP65UXhBd94F/pmKcEyY2PLSVQN3JZApj4jEWMZ1lcDj8kAAZG7759/MSfgnriOUPVHNzgazcTxJKtkJUrEvp8j9OG/zRSuW/onge/IH2iJy+ryiJgzlGMEAVkIYgXqLMH3wbyuGuJUIF6WBllgkNLgHNSLbk1C8WhC0lUwNDvhiFCwR5uAxSmUQ3oo0a4xjHfHGMoTtVvOP2KC2Ze8WWhcmkHt3QIdZMXG17l2EYC6nErJvyk2EQ8Rji7cd+/tPOaR/dLrvsW/TkItLdjOPDEukfK8Vman8fh0PNaC05S8kocEHUQhjsTdF8T71ZcOf9dAXLCSdGxQV6bhPtN912TMM+mVGqiG61uj+51xpHeWubeig8h5dETOmf2DXv77fWd04xgmJVR4cE5z+zCIbwofcxAyfWA+Krwav35A07nzwj5PTSX8YoVSL/Qu3JDomjnXyzXS2Nid6O9NZ78FZH1Wm6BXWW3YUpIIYjKgFADZ9w2FJIDvEoX7hj5kkB5vvmQfEQ3po01apLgyMdDoY8IhCbaCYpWafd3JDEG5acWmbqfmXaxBq4cpBqGwhwuFbc5KP8+Gi5ZFfnkXorosyOJC+lI8cnrtTSzHGUGJKs6UHTx3xD/c/M2ROnLKDShc4ONkSYxQhWW4ZUyajQRfmvTtu125pD6UsmUTaCvUIzsaAxdP0ac78s9nb1jFdfNzruKqzSRIer9AJF2kq0V9bnp+15nLosR59rX7Xyrr6XuvwyewVtdJngpkKqH8ztWRXrLTno5sLZK1/XdQMQp7qhcLcB8vBj7O4BhWYxUZ+XiDbjJ2q9jXqJufvRA6JjBt4JB4KZB6lquw3OeUD0e3TzR/apm2v9Oh9MlPvFay6IExOoBIBADVYnkOs4mTwJViYEpvqnj41asJ16Jb8/k3neUTwirOTYNusmjg5VTam5TS6yy2L+RwT6l1230022fPFzbO3UsMnGfwieSK5NtoNLkwC6l7FF0LaHp6TbcmJRaWqmVy+K3Mj3MNhWht41/GgbYDVBTQ6CuJF81i1JHxUMzHoS8Ju0noQDLizHgCI+9fIIqpmf4wUTJdUjHporDPYdpToKJBKQVRGAjfJAcmuN72w1q7loGQIklpHMcwcQH557squjadiu0j+4sdNHn2BbYXh/eSALUmO1ixuyvzDhZuV10u9QwlcEsLCHDblil5V9mBH2M2Igy+2HX/E93ZqTNcgnimJcjEB8uN/44mqbJnfM8kgE0aw0T0xiHj8LGcymm2L+47OQ7YtA7TTkKP69ng75dfqzzlhq9P5VE2HIOGa5K8ceZIAOF8uv3jsfRS2bUf1h9FoZBisXm2p2uR0wcAtcuymPOD+kXT6UYYsaHXIVQqesv4MBpqi7Y55meuI4kQC7QvJAvWH++9rPuLNWYMsrKLY8Uypc6b3lZB9Ubpv4sPlPgmSKMQl0YLQoMNJFmPlpaG8wdR5aL2gyMtfmELDFcxiD3CagqVIEhLb4hNhoOChA7Vy3Geax0GTW3nWpbKuYoIXMcdRhlnr+ysBiddDWPMmk95m4A1wVwUQ4t70FsqsEEX0fOf5lQaakF/ZiExLXVhVUHLOiSQzfPPzZs0N21xvXIyht6sDRY4RRfiMynQZwql6meqrlQE2+4Al4W0KcTQFMvoKfCcFff+qLkVWchlUWzkFZdziSG2J+wbETOtYv5L5g0TsddN42l9ZPsDnzKeE2HFBOhS9smNSFM0CpTqGZGfy9ZJXpmf/qmT1DNRoCPLNSygfI+4mhCu1/dgDANyY4kRBupqb5A+mxSYeaTXQKNoqIhXRWkt1d7HrfkGqu+SCEd0QKbkZMDreY9grAg7c/pRlJ6bIlaIVfNu3hLSmRiQ3wu5tTdhaYcynyRF5eHdujzsNJyDMft0tdtVCq0WeN1eQaAuw2wRngpSu7eTnvDKRq3iCAdC5jMQrYvc7j10op7VOvJtNIcVDJaMwoTM2TN5rdDTLoH5PRMCKXCCEnblgOs8vkYnLEo6+03u0HGjqmBsHAv7FE/uBOnNs6a/FwyojulecYYxc/StsFb2/Rd6Pu5USvE8kx6p6f/wmj4Mbt86HaP7yvAMMYqZDJ42EnYUjDabPy8hMfJvIvLpIBdzame24H07vnSrkAFJI7TQyBOohv0VwmxntVO1JFiJl88+j2Zvt/iRg43AFeqZt9O8JCEw00vdY4p8r5LqcTj7+uMtlRfOpMkO6XoZylBgX4jJZr++wGqlsIVP3W2+Zeem0ByLYH1RT7cIsseB6M83yO5LeiGGe0ZEaer396wTvytLst0lgfr4x/uTx5mYinhZk7ce6iI7uUfX/VVvSju5+CmzXLC4eow/ibmseYNM7VnWKfwMVHRqJH7uxcb4wP78Ii+fURiMDHm2seU/msmd6RHBOWhN3iRG4KZrPr6QRuYv3sXXwBTCnNkNCbPaU+KBHMvzF4e2g+oQet1r7Wdg/BWNI7QpTlgJKykc97/4AH5veNyCDcdg0v+RrFc3YffSjRhcWc2G8v1XMqQxYD+yWYlXJL9LZ57gTiERggMhCLSQ14BfRQ2WL6kk/d/yfAN0x9UjuWLxujzDS3O+H6+BijWPoSN3i9QHNZ/wYLH242QwQ2qeqlhe3hLpUFCeET74/V1yf6ngIDhzp1f9YgfCjz62iyFyQYri++BgrjJRGRMCGBTi87fOVapIGcc30JR6dw9TfFzDf23nZN/jdnRQbeXA556m6g9zv4VNax51hV3cfgQtHGPXmBfxbUfYpgp6Z/KggtzIN4Wp38nUZI6vT0VX9SvQu1BbSQ5uzxq3dpKfqxsMSrmWx2hoGPyX1R6MvFdG/7tk4waO8AgrVdBTRBN4EBMwKSTClDHuq02VtIm4kljT4F04O7plkJFlHePQApifeSGP1fz/PBUcj//50X9BjiMiFtFDwTIqm87744hmPbepUhBjUrW0Dfn5ZzJzX9NONAV6dQG3nT0ttQjRwCAiTI/S3ooN61WYFg5PcQ6raA0dA9w4p5hP7Ojcp4spGSRyqawB0k3bGdmIuRJfiUc6EvL6Cc3skdT0q2NCjAD0GqqOyFQGzkGXlhmHBGvMrLRBU0ar+WY8xQEZUrB8zzqLeQNLu+UdgscCHWU5ZZH0w/BXJzSCAFOM01fVgjK4UTX4chN5JsZDrsZhwCXN2EDjWrd2ku0GsCDFa3z8Y0TMZ5gtLPE9FAakQakBN7EdB3gOHeC6mU/tAARl4xD6MQmYshEWiJuQxPuAG/nCHVquFruR0QYBPT3KmOzbaOQHwAhrGmqOTp1sQ+beAo0oCrCRtZIAWBIj/mPyQp8hGpg2/oUgY6qybknQhrNonoekdtPqZkyOyaWIkEIiODFNyZ1rcCaJXBF50Y4GfU+Om1Y8BqWwvlge+Ya4YTpP9YX3qyprkQt++VQhgx1lYtjcr6W8wspunSJ0G0IjIXP6C8AV3A+Au/snEXv+46K7ZA4NWFALiT8kNgkMg6NFdXTHfg2S7YkklohVWSw/jT8RgKib1Raz/IqWgR2OPdFC8oEeOzHMiNF3T2/UzS3u7bSmlOpbhe5CXhQzsulWRH6sgAAAACJmlk0WAoF+gAB0lOAkAMA6oF32LHEZ/sCAAAAAARZWg=="

In [ ]:
step("test de l'accès Internet")
for url in ["https://huggingface.co", "https://pypi.org", "https://download.pytorch.org/whl/cu126/"]:
    code = subprocess.run(["curl", "-s", "-o", "/dev/null", "-m", "15", "-w", "%{http_code}", url],
                          capture_output=True, text=True).stdout or "000"
    print(f"  {url}: {code}")
    if code == "000":
        raise SystemExit("Pas d'accès Internet : activer 'Internet' dans les paramètres du notebook, puis relancer.")

In [ ]:
step("extraction du code, du lot et de la MSA intégrés")
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:xz") as tar:
    tar.extractall(WORK)
os.chdir(WORK)
step("téléchargement de la MSA des validateurs")
import hashlib
msa = WORK / "data/msa/" / MSA_NAME
for url in MSA_URLS:
    subprocess.run(["curl", "-sSL", "-m", "120", "--retry", "2", "-o", str(msa), url])
    if msa.exists() and hashlib.sha256(msa.read_bytes()).hexdigest() == MSA_SHA256:
        print(f"  MSA OK depuis {url}")
        break
    print(f"  échec depuis {url}")
else:
    raise RuntimeError("MSA introuvable ou corrompue")
sh("find . -type f | sort")
step("téléchargement de la base combinatoire (~120 Mo)")
sh('wget -nv --timeout=60 --tries=3 -O data/molecules.sqlite '
   '"https://huggingface.co/datasets/Metanova/Mol-Rxn-DB/resolve/main/molecules.sqlite"')
assert (WORK / "data/molecules.sqlite").stat().st_size > 100_000_000, "base incomplète"
step("étape 1 terminée")

## Environnement isolé pour Boltz 2.2.1
Boltz 2.2.1 exige `numpy<2` et Python ≤ 3.12 : il est installé dans un venv séparé (~2 min).

In [ ]:
step("installation de uv + Python 3.12")
sh(f"{sys.executable} -m pip install -q uv")
sh(f"uv venv -q --python 3.12 {VENV}")
py = str(VENV / "bin/python")
step("installation de torch 2.7.1 CUDA")
sh(f"uv pip install -q --python {py} torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126")
step("installation de boltz 2.2.1")
sh(f'uv pip install -q --python {py} boltz==2.2.1 "pyarrow<17" pandas requests pyyaml')
sh(f'{py} -c "import torch, boltz; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"')
step("étape 2 terminée")

## Précision de calcul
Les validateurs tournent en `bf16-mixed` sur des GPU récents. Les T4 (architecture 7.5) ne gèrent pas le bf16 : on passe Boltz en 32 bits. Léger écart, mesuré par les molécules de calibration du lot.

In [ ]:
cap = sh(f"{py} -c \"import torch; print('%d.%d' % torch.cuda.get_device_capability(0))\"").strip()
n_gpus = int(sh(f'{py} -c "import torch; print(torch.cuda.device_count())"'))
main_py = pathlib.Path(sh(f'{py} -c "import boltz.main as m; print(m.__file__)"').strip())
if float(cap) < 8.0:
    main_py.write_text(main_py.read_text().replace('"bf16-mixed"', '32'))
    print("bf16 non supporté -> précision 32 bits")
EXTRA = f"--no_kernels --devices {n_gpus if MULTI_GPU else 1}"
step(f"réglages Boltz : capacité {cap}, {EXTRA}")

## Scoring
Le premier lancement télécharge les poids Boltz-2 (~6 Go). Une ligne s'affiche à chaque molécule terminée (avec le temps restant estimé) et toutes les 10 minutes sinon. Relancer avec le même `RUN` reprend là où le calcul s'était arrêté.

In [ ]:
import pandas as pd
BATCH = "inputs/kaggle_batch.csv"
n_total = len(pd.read_csv(BATCH).head(TOP) if TOP else pd.read_csv(BATCH))
cmd = [py, "-m", "scripts.boltz_score", "--file", BATCH, "--accelerator", "gpu", "--run", RUN,
       "--boltz", str(VENV / "bin/boltz"), f"--extra={EXTRA}"] + (["--top", str(TOP)] if TOP else [])
log_path = OUT / f"boltz_{RUN}.log"
pred_dir = WORK / f"data/boltz/{RUN}/out/boltz_results_inputs/predictions"

def last_log_line():
    lines = [l.strip() for l in log_path.read_text(errors="replace").replace("\r", "\n").splitlines() if l.strip()]
    return lines[-1][:200] if lines else "(journal vide)"

step(f"lancement de Boltz sur {n_total} molécules")
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    done, first_done_at, last_report = 0, None, time.time()
    while proc.poll() is None:
        time.sleep(30)
        now_done = len(list(pred_dir.glob("*/affinity_*.json")))
        if now_done > done:
            first_done_at = first_done_at or time.time()
            rate = (time.time() - first_done_at) / 60 / (now_done - 1) if now_done > 1 else None
            eta = f" | {rate:.1f} min/molécule, ~{rate * (n_total - now_done):.0f} min restantes" if rate else ""
            step(f"{now_done}/{n_total} molécules notées{eta}")
            done, last_report = now_done, time.time()
        elif time.time() - last_report > 600:
            step(f"en cours ({done}/{n_total}) — {last_log_line()}")
            last_report = time.time()

print(log_path.read_text(errors="replace").replace("\r", "\n")[-4000:])
step(f"Boltz terminé (code {proc.returncode})")
assert proc.returncode == 0, "Boltz a échoué : voir le journal ci-dessus"

In [ ]:
import shutil
out = OUT / f"scores_{RUN}.csv"
shutil.copy(WORK / f"data/boltz/{RUN}/scores.csv", out)
df = pd.read_csv(out)
step(f"résultats : {out} ({len(df)} molécules)")
df